# **Phase 1: EXTRACT**

In [31]:
import pandas as pd
import sqlite3

In [32]:
df_raw = pd.read_csv('raw_ecommerce_data.csv')


In [33]:
df_raw.info()
df_raw.head()

<class 'pandas.DataFrame'>
RangeIndex: 185 entries, 0 to 184
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Order_ID       185 non-null    str  
 1   Customer_Name  183 non-null    str  
 2   Email          184 non-null    str  
 3   Product        185 non-null    str  
 4   Category       184 non-null    str  
 5   Order_Date     185 non-null    str  
 6   Quantity       185 non-null    int64
 7   Unit_Price     185 non-null    str  
 8   Amount         142 non-null    str  
dtypes: int64(1), str(8)
memory usage: 28.2 KB


,Order_ID,Customer_Name,Email,Product,Category,Order_Date,Quantity,Unit_Price,Amount
0,ORD-0036,Emma Brown,emma.brown@email.com,Monitor 24 inch,Electronics,24/04/2026,1,4131.00,"4,131.00"
1,ORD-0076,linda park,linda.park@email.com,Mechanical Keyboard,Electronics,"Apr 03, 2026",5,1669.50,8347.50
2,ORD-0101,john doe,john@email.com,Wireless Mouse,Electronics,11/04/2026,3,405.00,NaN
3,ORD-0059,Jane Smith,jane@email.com,Wireless Mouse,ELECTRONICS,13/03/2026,2,405.00,฿810.00
4,ORD-0038,PETER KIM,peter.kim@email.com,Gel Pen Set,Stationery,2026-02-28,5,85.50,427.50


# **Phase 2A: TRANFORM (สร้าง Dimension Table)**

In [34]:
dim_customer = df_raw[['Customer_Name', 'Email']].drop_duplicates()

dim_customer = dim_customer.reset_index(drop=True)
dim_customer['customer_id'] = dim_customer.index + 1

dim_customer = dim_customer[['customer_id', 'Customer_Name', 'Email']]

# **Phase 2B:TRANFORM (สร้าง Fact Table)**

In [35]:
fact_sales = pd.merge(df_raw, dim_customer,
                      on=['Customer_Name', 'Email'],
                      how='left')

fact_sales = fact_sales.drop(columns=['Customer_Name', 'Email'])

# ทำซ้ำกระบวนการนี้กับ Product และ Time Dimensions

In [36]:
dim_product = fact_sales[['Product']].drop_duplicates().reset_index(drop=True)
dim_product['product_id'] = dim_product.index + 1
dim_product = dim_product[['product_id', 'Product']]

fact_sales = pd.merge(fact_sales, dim_product, on=['Product'], how='left')

fact_sales = fact_sales.drop(columns=['Product'])

# **Phase 3A: LOAD (ตั้งค่า SQLite Warehouse)**

In [37]:
conn = sqlite3.connect('warehouse.db')
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS dim_customer (
        customer_id INTEGER PRIMARY KEY,
        Customer_Name TEXT,
        Email TEXT
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS dim_product (
        product_id INTEGER PRIMARY KEY,
        Product TEXT
    )
''')
conn.commit()

# **Phase 3B: LOAD (บังคับใช้ Star Schema Relationships)**

In [38]:
cursor.execute('PRAGMA foreign_keys = ON;')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS fact_sales (
        transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
        customer_id INTEGER,
        product_id INTEGER,
        Amount REAL,
        FOREIGN KEY (customer_id) REFERENCES dim_customer(customer_id),
        FOREIGN KEY (product_id) REFERENCES dim_product(product_id)
    )
''')
conn.commit()

# **Phase 3C: LOAD (ผลักข้อมูลลง Warehouse)**

In [39]:
dim_customer.to_sql('dim_customer', con=conn, if_exists='replace', index=False)
dim_product.to_sql('dim_product', con=conn, if_exists='replace', index=False)

fact_sales.to_sql('fact_sales', con=conn, if_exists='replace', index=False)

print('ETL Pipeline ran successfully!')

conn.close()

ETL Pipeline ran successfully!


# **Verification: ทดสอบคิวรีจาก Warehouse**

In [40]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('warehouse.db')

sql_query = """
SELECT
    c.Customer_Name,
    SUM(f.amount) as Total_Spend
FROM fact_sales f
JOIN dim_customer c ON f.customer_id = c.customer_id
GROUP BY c.Customer_Name
ORDER BY Total_Spend DESC
LIMIT 3;
"""

result_df = pd.read_sql_query(sql_query, conn)

print("--- ผลลัพธ์การคิวรีข้อมูล (Top 3 Customers) ---")
print(result_df)

conn.close()

--- ผลลัพธ์การคิวรีข้อมูล (Top 3 Customers) ---
  Customer_Name  Total_Spend
0     Narin Dee      37597.5
1    Jane Smith      26985.0
2    Alice Wong      25009.5
